# ScientificLoop: GRPO Training Notebook\n",
    "\n",
    "> **Teaching an LLM to reproduce scientific papers through reinforcement learning.**\n",
    "\n",
    "This notebook launches GRPO training for **Qwen2.5-Coder-7B-Instruct** with LoRA on HuggingFace's A10G GPU via `hf jobs`. Training runs entirely on HF's servers — logs stream into the cell and the job keeps going even if you close the tab.\n",
    "\n",
    "**What this notebook does:**\n",
    "1. Installs dependencies + authenticates with HuggingFace\n",
    "2. Launches training on **A10G (24GB VRAM)** via `hf jobs uv run`\n",
    "3. Streams live logs into the cell\n",
    "4. Plots final training curves from saved metrics\n",
    "5. Runs inference on the trained model\n",
    "\n",
    "**Trained model:** [Sushant0809/scientific-loop-grpo](https://huggingface.co/Sushant0809/scientific-loop-grpo)\n",
    "\n",
    "---\n",
    "[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sushant0809/scientific-loop/blob/main/train_grpo.ipynb)"

## Step 1: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch runtime to GPU')

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install -q \
    "trl>=1.2.0" \
    transformers \
    accelerate \
    datasets \
    peft \
    "huggingface_hub[cli]" \
    "openenv-scientific-loop @ git+https://huggingface.co/spaces/Sushant0809/scientific-loop"

print('Dependencies installed.')

## Step 3: Authenticate with HuggingFace

Needed to push the trained LoRA adapter to your Hub repo. Set `PUSH_TO_HUB = False` in Step 4 to skip.

In [ ]:
import os
from huggingface_hub import login

# ── Get HF token from Colab Secrets (recommended) ────────────────────────────
# Add your token: left sidebar → 🔑 Secrets → "+ Add new secret"
# Name: HF_TOKEN   Value: hf_...
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    # Fallback: set manually or via environment variable
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ Logged in to HuggingFace.')
else:
    print('⚠ No HF_TOKEN found. Set it in Colab Secrets (🔑) or paste below:')
    # HF_TOKEN = 'hf_...'   # ← uncomment and paste token as fallback

## Step 4: Configuration

In [ ]:
import os

# ── Change these to match your setup ─────────────────────────────────────────
MODEL_NAME     = 'Qwen/Qwen2.5-Coder-7B-Instruct'  # use 1.5B variant for T4
OUTPUT_DIR     = './outputs/scientific-loop-grpo'
HUB_MODEL_ID   = 'Sushant0809/scientific-loop-grpo' # your HF repo
PUSH_TO_HUB    = bool(HF_TOKEN)
TOTAL_EPISODES = 200
NUM_EPOCHS     = 2    # 100 steps ≈ 2.5hr on A10G
EVAL_EVERY     = 25   # eval callback frequency (steps)

# Fix CUDA memory fragmentation before torch import
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model      : {MODEL_NAME}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Push to Hub: {PUSH_TO_HUB}')
print(f'Total steps: {TOTAL_EPISODES // 4 * NUM_EPOCHS}')

## Step 5: Load Model + Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model (bfloat16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
)
print('Model loaded. LoRA r=8 config ready (~80M trainable params).')

## Step 6: Dataset + Reward Function

The dataset is curriculum-ordered: warmup papers first (pure numpy), then gradually harder.

The reward combines:
- **Execution reward** — metric proximity + execution quality
- **Format reward** — syntax check + METRICS line detection (no code execution needed)

The format reward prevents **dead batches** (zero reward variance → zero GRPO gradient) by providing signal even when all completions fail at runtime.

In [ ]:
import json
import re
from datasets import Dataset
from ScientificLoop.paper_corpus import EVAL_PAPERS, format_paper_for_agent, load_paper, sample_paper
from ScientificLoop.server.execution_engine import compute_metric_proximity, extract_metrics, run_code
from ScientificLoop.reward_calculator import compute_format_reward, compute_step_reward

SYSTEM_PROMPT = (
    'You are an expert ML engineer. '
    'Read the paper methodology below and implement it as a complete, '
    'self-contained Python script. Output ONLY executable Python code — '
    'no markdown, no explanation, no code fences. '
    'The script must print its results on the last line as:\n'
    'METRICS: {"metric_name": value}'
)

def make_dataset(total_episodes=TOTAL_EPISODES):
    rows = []
    for ep in range(total_episodes):
        paper = sample_paper(episode_number=ep)
        rows.append({
            'prompt': f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(paper)}',
            'paper_id': paper.paper_id,
            'episode': ep,
            'difficulty': paper.difficulty,
        })
    return Dataset.from_list(rows)

def _extract_code(raw):
    raw = raw.strip()
    fence = re.search(r'```(?:python)?\n(.*?)```', raw, re.DOTALL)
    if fence:
        return fence.group(1).strip()
    raw = re.sub(r'```(?:python)?', '', raw)
    return re.sub(r'```', '', raw).strip()

def reward_fn(prompts, completions, **kwargs):
    paper_ids = kwargs.get('paper_id', [None] * len(completions))
    rewards = []
    for code, paper_id in zip(completions, paper_ids):
        paper = load_paper(paper_id)
        code = _extract_code(code)
        stdout, stderr, timed_out = run_code(code, paper.execution_timeout)

        if 'SecurityError' in stderr: exec_status = 'blocked'
        elif timed_out:               exec_status = 'timeout'
        elif stderr and not stdout:   exec_status = 'error'
        else:                         exec_status = 'success'

        achieved = extract_metrics(stdout)
        score, _ = compute_metric_proximity(achieved, paper.target_metrics, paper.metric_weights)
        matched = sum(
            1 for m, tv in paper.target_metrics.items()
            if m in achieved and abs(achieved[m] - tv) / max(abs(tv), 1e-6) <= 0.10
        )
        exec_reward = compute_step_reward(
            reproduction_score=score, prev_reproduction_score=0.0,
            execution_status=exec_status, step_number=1,
            current_code=code, previous_code=None,
            metrics_matched_count=matched, total_metrics=len(paper.target_metrics),
        )
        fmt_reward = compute_format_reward(code)
        rewards.append(round(exec_reward + 0.5 * fmt_reward, 4))
    return rewards

train_dataset = make_dataset()
eval_dataset  = Dataset.from_list([
    {'prompt': f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(p)}', 'paper_id': p.paper_id}
    for p in EVAL_PAPERS
])
print(f'Train: {len(train_dataset)} episodes  |  Eval: {len(eval_dataset)} papers')

## Step 8: Launch Training on HuggingFace A10G\n",
    "\n",
    "This submits the job to HuggingFace's A10G hardware (24GB VRAM). Training runs on HF's servers and logs stream directly into this cell. Your tab can close and training keeps going."

In [ ]:
import os
import subprocess
from google.colab import userdata

# Make HF_TOKEN available to the hf CLI
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or HF_TOKEN

result = subprocess.run(
    [
        'hf', 'jobs', 'uv', 'run',
        '--with', 'trl>=1.2.0',
        '--with', 'torch',
        '--with', 'transformers',
        '--with', 'accelerate',
        '--with', 'datasets',
        '--with', 'peft',
        '--with', 'openenv-scientific-loop @ git+https://huggingface.co/spaces/Sushant0809/scientific-loop',
        '--flavor', 'a10g-small',
        '-s', 'HF_TOKEN',
        '--', 'python', 'train_grpo.py',
    ],
    capture_output=False,  # stream logs directly into this cell
    text=True,
)

if result.returncode == 0:
    print('\n✅ Training completed successfully.')
else:
    print(f'\n❌ Job exited with code {result.returncode}')

## Step 9: Save the Final Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')
if PUSH_TO_HUB:
    print(f'Pushed to: https://huggingface.co/{HUB_MODEL_ID}')

## Step 10: Final Training Summary Plot

Renders the complete training history as a static, publication-quality figure and saves it to `training_curves.png`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

steps       = train_log['step']
rew_mean    = train_log['reward_mean']
rew_std     = train_log['reward_std']
dead        = train_log['dead_frac']
comp_len    = train_log['completion_len']
eval_steps  = eval_log['step']
eval_scores = eval_log['score']

def smooth(v, w=5):
    if len(v) < w: return np.array(v)
    return np.convolve(v, np.ones(w)/w, mode='valid')

fig = plt.figure(figsize=(15, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

best_reward = max(rew_mean) if rew_mean else 0
fig.suptitle(
    f'ScientificLoop GRPO Training  |  {MODEL_NAME.split("/")[-1]} + LoRA r=16\n'
    f'{len(steps)} steps  ·  {NUM_EPOCHS} epochs  ·  best reward: {best_reward:.3f}',
    fontsize=13, fontweight='bold', y=1.01
)

# ── 1. Mean Reward ───────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, rew_mean, color='#aecbfa', alpha=0.4, linewidth=1, label='raw')
sm = smooth(rew_mean)
ax1.plot(steps[len(steps)-len(sm):], sm, color='#1a73e8', linewidth=2.5,
         label=f'smoothed (w=5)')
ax1.axhline(-2.1, color='#d93025', linestyle='--', linewidth=1.2,
             label='error floor (−2.1)')
ax1.fill_between(steps, rew_mean, -2.1,
                 where=[r > -2.1 for r in rew_mean],
                 alpha=0.15, color='#34a853', label='above floor')
ax1.set_title('Mean Reward per Step', fontweight='bold')
ax1.set_xlabel('Training Step'); ax1.set_ylabel('Mean Reward')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# ── 2. Reward Std ────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(steps, rew_std, color='#fbbc04', alpha=0.5, width=0.8, label='raw')
sm2 = smooth(rew_std)
ax2.plot(steps[len(steps)-len(sm2):], sm2, color='#e37400', linewidth=2,
         label='smoothed')
ax2.set_title('Reward Diversity (std > 0 → gradient signal)', fontweight='bold')
ax2.set_xlabel('Training Step'); ax2.set_ylabel('Reward Std')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# ── 3. Dead Batches ──────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
colors = ['#34a853' if v == 0 else ('#fbbc04' if v < 1 else '#ea4335') for v in dead]
ax3.bar(steps, dead, color=colors, alpha=0.8, width=0.8)
ax3.axhline(1.0, color='#d93025', linestyle='--', linewidth=1, label='all dead (old baseline)')
pct = 100 * np.mean([v > 0 for v in dead]) if dead else 0
ax3.set_title(f'Dead Batches  ({pct:.0f}% had zero std  ·  green = gradient flowing)',
              fontweight='bold')
ax3.set_xlabel('Training Step'); ax3.set_ylabel('Fraction zero-std')
ax3.set_ylim(0, 1.15); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── 4. Eval Score + Completion Length ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4b = ax4.twinx()
if comp_len:
    ax4b.fill_between(steps, comp_len, alpha=0.12, color='#9aa0a6')
    sm_len = smooth(comp_len)
    ax4b.plot(steps[len(steps)-len(sm_len):], sm_len, color='#5f6368',
              linewidth=1.5, linestyle='--', label='mean completion len')
    ax4b.set_ylabel('Completion Length (tokens)', color='#5f6368', fontsize=9)
if eval_steps:
    ax4.plot(eval_steps, eval_scores, color='#a142f4', linewidth=2.5,
             marker='o', markersize=7, zorder=5, label='eval repro score')
    ax4.axhline(0.8, color='#34a853', linestyle=':', linewidth=1.5, label='target (0.80)')
    for s, v in zip(eval_steps, eval_scores):
        ax4.annotate(f'{v:.3f}', (s, v), textcoords='offset points',
                     xytext=(0, 8), fontsize=8, color='#a142f4', ha='center')
ax4.set_title('Eval Reproduction Score + Completion Length', fontweight='bold')
ax4.set_xlabel('Training Step'); ax4.set_ylabel('Reproduction Score (0–1)')
ax4.set_ylim(-0.02, 1.05)
lines1, lbl1 = ax4.get_legend_handles_labels()
lines2, lbl2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, lbl1 + lbl2, fontsize=8)
ax4.grid(alpha=0.3)

plt.savefig(f'{OUTPUT_DIR}/training_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR}/training_curves_final.png")

# Print summary table
print(f"\n{'─'*50}")
print(f"  Training Summary")
print(f"{'─'*50}")
print(f"  Total steps        : {len(steps)}")
print(f"  Best mean reward   : {max(rew_mean):.4f}" if rew_mean else "  No data")
print(f"  Avg reward std     : {np.mean(rew_std):.4f}" if rew_std else "")
print(f"  Dead batches (%)   : {100*np.mean([v>0 for v in dead]):.1f}%" if dead else "")
if eval_steps:
    print(f"  Eval checkpoints   : {len(eval_steps)}")
    print(f"  Best eval score    : {max(eval_scores):.4f}")
    print(f"  Last eval score    : {eval_scores[-1]:.4f}")
print(f"{'─'*50}")

## Step 11: Inference Test

Try the trained model on all warmup papers to verify it generates runnable code.

In [ ]:
from ScientificLoop.paper_corpus import WARMUP_PAPERS

print('=== Inference on Warmup Papers ===\n')
for paper in WARMUP_PAPERS:
    prompt = f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(paper)}'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    del out, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    code = _extract_code(generated)
    stdout, stderr, timed_out = run_code(code, paper.execution_timeout)
    achieved = extract_metrics(stdout)
    score, deltas = compute_metric_proximity(achieved, paper.target_metrics, paper.metric_weights)

    status = 'TIMEOUT' if timed_out else ('ERROR' if stderr and not stdout else 'OK')
    print(f"[{paper.paper_id}]")
    print(f"  Status  : {status}")
    print(f"  Score   : {score:.3f}")
    print(f"  Achieved: {achieved}")
    print(f"  Target  : {paper.target_metrics}")
    print(f"  Deltas  : {deltas}")
    if stderr: print(f"  Stderr  : {stderr[:120]}")
    print()

## Links

- **Environment (HF Space):** https://huggingface.co/spaces/Sushant0809/scientific-loop
- **Trained Model (LoRA adapter):** https://huggingface.co/Sushant0809/scientific-loop-grpo
- **Blog post:** [ScientificLoop: Teaching an LLM to Reproduce Scientific Papers with RL](https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/hf_blog_post.md)

---
*OpenEnv Hackathon — PyTorch × Scaler — April 2026*